In [0]:
# Part 1 — the rate Source
# Q1. Create a streaming DataFrame using rate, 5 rows/sec.

#simulate a live source of data
df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

#check if is streaming
df.isStreaming

True

In [0]:
#Q2. Start the query writing to an in-memory sink. What outputMode?

query = (
    df.writeStream
    .outputMode("append")
    .format("memory")
    .queryName("rate_demo")
    .option("checkpointLocation","/Volumes/workspace/default/data_upload/checkpoint_rate")
    .trigger(availableNow=True)
    .start()
)

In [0]:
spark.table("rate_demo").show()

+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+



In [0]:
#Q3. Query rate_demo twice. What changes, and what does it show?

spark.table("rate_demo").count()

# Problem: databricks does not support continuously running streaming queries on its serverless cloud infrastructure. (solution -> .trigger in Q2)

# Answer:  But due to the free version of Databricks there are limitations on streaming data and to the fix for problem (.trigger(availableNow=True)) triggers the processes and stops, so the count remains unchanged. 
# However the row count would be higher on the second query what would demonstrate that the streaming query executes as a continuous background micro-batch job, independent of the notebook cell that started it.


0

In [0]:
# Part 2 — File-Based Streaming

# Q4. Create a Volume as the landing zone.
#"input" folder created via databricks UI 
# -> Catalogs -> workspace -> default -> Volumes ->data_upload -> input


In [0]:
#Q5. Define a schema. Why explicit, not inferSchema?

schema = "id INT, name STRING, amount DOUBLE, event_time TIMESTAMP"

#Answer: Structured Streaming processes files incrementally, one micro-batch at a time. inferSchema would require scanning all data up front to infer types, which is incompatible with an unbounded, incrementally-arriving stream — Spark disallows it for streaming file sources and requires an explicit schema.


In [0]:
#Q6. Streaming read of CSVs from the landing folder.

stream_df = (
    spark.readStream
    .format("csv")
    .schema(schema)
    .option("header", "true")
    .load("/Volumes/workspace/default/data_upload/input/")
)

In [0]:
# Part 3 — Writing to Delta
# Q7. Write to a Delta table with a checkpoint. Why mandatory?

query = stream_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/data_upload/checkpoint_csv") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("workspace.default.streaming_output")

# Answer: The checkpoint stores processed file offsets and state. Without it Spark can't guarantee exactly-once processing or resume correctly after a restart/failure — it would either reprocess files (duplicates) or lose track of progress entirely.


In [0]:
# Q8. Drop 2-3 CSVs into the folder one at a time; query the output table after each.

spark.table("workspace.default.streaming_output").show()

# Answer: Row count in workspace.default.streaming_output increases after each file lands, but only once the micro-batch trigger fires (default trigger runs as fast as possible, typically a few seconds). This shows incremental, append-only ingestion driven by new files rather than a fixed schedule.
# In Databricks Free Edition, availableNow=True processes all currently available files and then stops. Therefore, I must rerun the streaming query after uploading each new file.
# The checkpoint remembers which files were already processed and prevents them from being added again.

In [0]:
#Q9. display() on the streaming DataFrame directly. How does it differ?

display(
    stream_df,
    checkpointLocation="/Volumes/workspace/default/data_upload/checkpoint_display"
)
# Answer: display(stream_df) creates a streaming dashboard directly in the notebook. It shows processing metrics and the incoming data without writing to a Delta table. It runs as a separate streaming query and requires its own checkpoint.

Checkpointing to /Volumes/workspace/default/data_upload/checkpoint_display


Q10. File simulation vs. a true source like Kafka.

File-based streaming processes new files added to a folder. It is useful for testing but is usually slower and depends on files being completely uploaded before Spark can read them.

Kafka streams individual messages in near real time.It supports multiple consumers and allows messages to be replayed.

Q11. What changes moving Free Edition to production with a real source?

Answer: Swap .format("csv")/.load(path) for .format("kafka") (or the relevant connector) with broker/topic options; typically also move to a job cluster or a scheduled/continuous Job instead of an interactive notebook, and add monitoring/alerting on the streaming query via the Spark UI or system tables.

1-Replace the CSV file source with a real streaming source, such as Kafka:<br>

this lab: reads CSV files from a folder: spark.readStream.format("csv")<br>
in production: reads messages from a Kafka topic: spark.readStream.format("kafka")<br>

2-Configure the Kafka broker address and topic.<br>
3-Run the pipeline as a continuously running Databricks job.<br>
4-add monitoring and alerts to detect failures.